<a href="https://colab.research.google.com/github/TurkuNLP/intro-to-nlp/blob/master/ex9_word2vec_solved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Exercise task 9: word2vec

## Part A

**Figure out whether in general higher similarity is given to semantically related words (e.g. Volvo, Audi, Nissan) or different inflections of same lemma (e.g. drive, drives, driving), or a mixture of both.**

Based on the online demo, it seems to be a mixture of both, but maybe semantically related words are slightly more common...?
For example,
```
dog --> dogs, puppy, pit_bull, pooch, cat, golden_retriever, German_shepherd
Volvo --> Volkswagen, VW, Volvo_Cars, BMW, Audi, Nissan
sprint --> sprints, sprinting, sprinters, sprinter, criterium, race
apple --> apples, pear, fruit, berry, pears, strawberry
Apple --> Apple_AAPL, Apple_Nasdaq_AAPL, Apple_NASDAQ_AAPL, Apple_Computer, iPhone, Steve_Jobs
```
Sometime the inflection itself is a very strong feature:
```
faster --> quicker, slower, speedier, Faster, smoother, swifter
kävelin (in Finnish model) --> astelin, kipitin, juoksin, hölkkäsin, kiiruhdin, suunnistin
```
* This can be seen as e.g. that comparatives tend to occur in similar contexts in general (e.g. "it is **better** than that", "it is **faster** than that"...)
* Also, in Finnish, verbs conjugated for person (e.g. _minä kävelen_, _sinä kävelet_), often occur in different contexts, since _sinä_ is unlikely to appear very close to _kävelen_.


**Try few ambiguous words (e.g. bank, rock) to see how these seem to be represented. Does the embedding reflect more one of the possible meanings? Then try "similarity of two words" to measure the actual similarities to synonyms of both possible meanings (e.g. rock – rock'n'roll / rock – stone). Do you observe a difference in the similarities?**
```
rock --> rock_n_roll, rockers, punk_rock, alt_rock, rocks, indie_rock (clearly music)
bank --> banks, banking, Bank, lender, banker, depositors (clearly financial)
orange --> bright_orange, purple, blue, yellow, bright_yellow, red (clearly color)
```
Looks like only one of the possible meanings are visible in top-6 lists. My hypothesis is that this is the most common meaning in the training data.

```
rock - rock_n_roll: 0.63223916
rock - stone: 0.3751019
rock - spaghetti: 0.15331538 (baseline to understand what similarity is to a random unrelated word)

orange - purple: 0.6877813
orange - apple: 0.39203462
orange - spaghetti: 0.22850524 (baseline to understand what similarity is to a random unrelated word, however, note that both are something you can eat!)
```
The similarity is clearly higher with a word related to the "primary" maining as observed above, however, word related to the alternative meaning get higher similarity compared to a random unrelated word.


## Part B

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.8 MB/s eta 0:00:00


In [ ]:
!wget http://dl.turkunlp.org/intro-to-nlp/enwiki-20220301-sample-tokenized.txt

--2026-04-15 19:42:16--  http://dl.turkunlp.org/intro-to-nlp/enwiki-20220301-sample-tokenized.txt
Resolving dl.turkunlp.org (dl.turkunlp.org)... 195.148.30.23
Connecting to dl.turkunlp.org (dl.turkunlp.org)|195.148.30.23|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 543383248 (518M) [text/plain]
Saving to: ‘enwiki-20220301-sample-tokenized.txt’

enwiki-20220301-sam 100%[===================>] 518.21M  11.0MB/s    in 40s     

2026-04-15 19:42:57 (13.0 MB/s) - ‘enwiki-20220301-sample-tokenized.txt’ saved [543383248/543383248]



The dataset contains 3.8 million sentences and 102 million tokens. Training word2vec on full data and 5 epochs takes about 22 minutes. Trim the data while debugging.

In [ ]:
# Let's make a list of sentences, each sentence being a list of tokens
# Data is pretokenized so we only need whitespace splitting

sentences = []
with open("enwiki-20220301-sample-tokenized.txt", "r") as f:
    for line in f:
        sentences.append(line.split())
print(f"Number of sentences: {len(sentences)}")
print(f"Number of tokens: {sum([len(s) for s in sentences])}")

Number of sentences: 3869427
Number of tokens: 102401825


In [ ]:
%%time

# %%time is colab thing to print out run time
# Note that here we use fairly small values for vector_size and window.
# It would make sense to investigate different values.
# In general, a small window tends to produce embeddings that capture more
# syntactic or functional similarity, whereas a larger window tends to
# capture broader semantic or topical similarity.

# train model
model = Word2Vec(
    sentences,
    sg=0,
    vector_size=40,
    window=2,
    min_count=1,
    workers=1,
    epochs=5
)

CPU times: user 22min 18s, sys: 6.42 s, total: 22min 24s
Wall time: 22min 26s


In [ ]:
# language modeling usage (given context, predict target word)
pred = model.predict_output_word(["Helsinki", "is", "of", "Finland"], topn=5)
for w,s in pred:
    print(f"{w}: {s:.6f}")

region: 0.000024
Biennale: 0.000023
University: 0.000020
part: 0.000020
headquartered: 0.000017


Maybe not great, but these are still understandable and not random.

In [ ]:
# nearest words using word embeddings
for word in ['good', 'bad', 'dog', 'cat', 'king', 'queen']:
  print(f'{word}:\t', [w for w, s in model.wv.most_similar(word)])
  print()

good:	 ['decent', 'bad', 'little', 'great', 'fair', 'real', 'nice', 'genuine', 'hard', 'big']

bad:	 ['good', 'tough', 'terrible', 'little', 'strange', 'decent', 'quiet', 'dangerous', 'crazy', 'horrible']

dog:	 ['cat', 'donkey', 'cow', 'baby', 'horse', 'monster', 'rabbit', 'creature', 'mask', 'boy']

cat:	 ['dog', 'rabbit', 'monkey', 'goat', 'snake', 'donkey', 'cow', 'wolf', 'monster', 'spider']

king:	 ['prince', 'emperor', 'monarch', 'queen', 'princess', 'sultan', 'duke', 'throne', 'kingdom', 'Emperor']

queen:	 ['king', 'princess', 'prince', 'knight', 'monarch', 'bride', 'consort', 'empress', 'demon', 'warrior']



Not too bad a result for a half-hour run! Some observations:

* Perhaps surprisingly, antonyms (words with opposite meaning) show up in the lists of most similar words. This is a known issue with word embeddings (see e.g. Scheible et al. 2013) and makes sense if you think about it: for example, bad can appear in many of the same contexts as good.
* These word embeddings group animal words with both dog and cat, but don't show the level of granularity that can be get with the embeddings trained on more data, where different types of dogs and cats were more similar to the words dog and cat (respectively) than other animals.
* We see a similar phenomenon with king and queen as with animals, with many words related to royalty and ruling classes showing up in both, with more limited distinction by e.g. gender than is often found in embeddings trained on more data.


In [ ]:
# word2vec TOY EXAMPLE
# NOTE: This is a toy model that only demonstrates how to train word2vec,
# and that it will learn very simple patterns.
# This model will not give reasonable embedding similarities because of lack of data.

from gensim.models import Word2Vec

sentence_a = ["the", "cat", "liked", "milk"]
sentence_b = ["the", "dog", "ate", "food"]

sentences = [sentence_a]*7000 + [sentence_b]*7000 # repeat the same toy sentences multiple times

# train model
model = Word2Vec(
    sentences,
    sg=0,
    vector_size=40,
    window=2,
    min_count=1,
    workers=1,
    epochs=100
)

# test target word prediction (language modeling objective)
for context in [["the", "liked", "milk"], ["the", "ate", "food"], ["the", "liked", "food"]]:
    print("Context:", context)
    pred = model.predict_output_word(context, topn=5)
    print("Top predictions:")
    for w,s in pred:
        print(f"{w}: {s:.2f}")
    print()

# the model is able to learn a very simple pattern, where "liked milk" predicts "cat", and "ate food" predicts "dog"
# For "liked food" context, dog/cat/milk/ate are equally likely

Context: ['the', 'liked', 'milk']
Top predictions:
cat: 0.68
milk: 0.11
liked: 0.10
the: 0.06
dog: 0.02

Context: ['the', 'ate', 'food']
Top predictions:
dog: 0.72
food: 0.09
ate: 0.09
the: 0.06
cat: 0.01

Context: ['the', 'liked', 'food']
Top predictions:
dog: 0.22
milk: 0.21
cat: 0.20
ate: 0.20
the: 0.13

